In [ ]:
import os
import pandas as pd

os.makedirs('data', exist_ok=True)

URL = 'https://datasets.imdbws.com/'
SOUTH_ASIA = ['IN', 'PK', 'BD']
WEST = ['US', 'GB', 'AU', 'CA', 'IE', 'NZ']

In [ ]:
akas = pd.read_csv(URL + 'title.akas.tsv.gz', sep='\t', na_values='\\N',
                   usecols=['titleId', 'region', 'isOriginalTitle'])

sa = set(akas[akas['region'].isin(SOUTH_ASIA)]['titleId'])
western = set(akas[(akas['region'].isin(WEST)) & (akas['isOriginalTitle'] == 1)]['titleId'])
ids = sa - western

In [ ]:
basics = pd.read_csv(URL + 'title.basics.tsv.gz', sep='\t', na_values='\\N',
                     usecols=['tconst', 'titleType', 'primaryTitle', 'startYear',
                              'runtimeMinutes', 'genres'])

movies = basics[(basics['titleType'] == 'movie') & (basics['tconst'].isin(ids))].copy()
movies = movies.drop(columns='titleType')
movies['startYear'] = pd.to_numeric(movies['startYear'], errors='coerce')
movies['runtimeMinutes'] = pd.to_numeric(movies['runtimeMinutes'], errors='coerce')

In [ ]:
ratings = pd.read_csv(URL + 'title.ratings.tsv.gz', sep='\t')
movies = movies.merge(ratings, on='tconst', how='left')

In [ ]:
roles = ['director', 'actor', 'actress', 'composer', 'writer', 'producer']

principals = pd.read_csv(URL + 'title.principals.tsv.gz', sep='\t', na_values='\\N',
                         usecols=['tconst', 'nconst', 'category'])
principals = principals[principals['tconst'].isin(movies['tconst']) &
                        principals['category'].isin(roles)]

names = pd.read_csv(URL + 'name.basics.tsv.gz', sep='\t', na_values='\\N',
                    usecols=['nconst', 'primaryName'])
principals = principals.merge(names, on='nconst', how='left')

In [ ]:
def join_names(cats):
    return (principals[principals['category'].isin(cats)]
            .groupby('tconst')['primaryName']
            .apply(lambda x: ', '.join(x.dropna())))

movies['directors'] = movies['tconst'].map(join_names(['director']))
movies['actors'] = movies['tconst'].map(join_names(['actor', 'actress']))
movies['music_director'] = movies['tconst'].map(join_names(['composer']))
movies['writers'] = movies['tconst'].map(join_names(['writer']))
movies['producers'] = movies['tconst'].map(join_names(['producer']))

In [ ]:
movies = movies[['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres',
                 'averageRating', 'numVotes', 'directors', 'actors',
                 'music_director', 'writers', 'producers']]
movies.to_csv('data/imdb_south_asian.csv', index=False)
movies.head()